# Env38 - Topological and Network Metrics

Professor-facing inspection of the accepted Env38 topology baseline using the tested production module.

## 1. Purpose and scope
Reproduce the stored topology calculation in read-only mode and explain its graph populations, configuration, results, provenance, and limitations.

## 2. Dynamic repository and accepted-run selection
Both `latest.json` pointers are resolved at execution time. No analysis run identifier is embedded in this notebook.

In [ ]:
from pathlib import Path
import json, sys

def find_repository(start):
    for candidate in (start, *start.parents):
        if (candidate / 'environment.yml').is_file() and (candidate / 'config' / 'environments.yaml').is_file():
            return candidate
    raise RuntimeError('Repository root not found')

REPOSITORY = find_repository(Path.cwd().resolve())
if str(REPOSITORY / 'src') not in sys.path:
    sys.path.insert(0, str(REPOSITORY / 'src'))
from geogami_morphology.environments import get_environment
from geogami_morphology.metrics_topology import analyze_topology, load_topology_config
ENVIRONMENT = 'env38'
CANONICAL_RUN = 'latest'
environment = get_environment(ENVIRONMENT, repository_root=REPOSITORY)
canonical_pointer = json.loads(environment.canonical_latest.read_text(encoding='utf-8'))
analysis_pointer_path = environment.analysis_root / 'latest.json'
analysis_pointer = json.loads(analysis_pointer_path.read_text(encoding='utf-8'))
assert canonical_pointer['run_id'] == analysis_pointer['canonical_run_id']
assert (REPOSITORY / analysis_pointer['canonical_path']).resolve() == (REPOSITORY / canonical_pointer['canonical_path']).resolve()
print('ENV38_TOPOLOGY_POINTER_RESOLUTION: PASS')

## 3. Graph representations and physical-edge semantics
The NetworkX `MultiGraph` contains each undirected physical edge once. The OSMnx-compatible `MultiDiGraph` contains a reciprocal arc pair for every physical edge; therefore 69 physical edges correspond to 138 directed arcs, not 138 streets.

## 4. Versioned metric configuration
The production loader verifies length weighting, connectedness policy, OSMnx intersection threshold, and centrality normalization from `config/metrics.yaml`.

In [ ]:
config = load_topology_config(project_root=REPOSITORY)
result = analyze_topology(ENVIRONMENT, canonical_run=CANONICAL_RUN, latest_path=environment.canonical_latest, project_root=REPOSITORY, publish=False)
assert result.selection.run_id == canonical_pointer['run_id']
assert result.physical_graph.number_of_nodes() == 46
assert result.physical_graph.number_of_edges() == 69
assert result.osmnx_graph.number_of_edges() == 138
print(f'TOPOLOGY_CONFIG: {config.path.relative_to(REPOSITORY)} | sha256={config.file_sha256}')
print('ENV38_TOPOLOGY_CALCULATION: PASS')

## 5. Controlled graph identity
Node count, physical-edge count, connected components, self-loops, and parallel edges document the topology held fixed by the experiment.

## 6. Degree and junction structure
Degree counts and proportions describe dead ends and three- and four-way junctions on the physical graph.

## 7. OSMnx cross-checks
OSMnx street-count and intersection summaries are cross-checks against the NetworkX physical-graph interpretation, not a second source network.

## 8. Cycle structure
Cycle rank records independent cycles and depends only on controlled connectivity.

## 9. Bridges, articulation points, and connectivity
These metrics characterize structural vulnerability on the validated simple view after proving that simplification collapses no loops or parallel physical edges.

## 10. Shortest paths: hops and local length
Hop distance is topology-controlled. Length-weighted paths use canonical local Cartesian edge length and are geometry-sensitive. GeoGami Local Cartesian coordinates are not longitude/latitude. Distances are reported in local units; do not call them metres unless a justified scale transformation is established.

## 11. Node and edge centrality
The published node and edge tables retain unweighted and local-length-weighted betweenness, closeness, eccentricity, and mean-distance detail.

## 12. Controlled versus geometry-weighted metrics
Connectivity-only values are topology controls. Values weighted by `length` may differ between Env38 and Env39 even when identifiers and connectivity agree.

## 13. Result table and stored-artifact cross-check
The following table is the production module result, checked field-for-field against the accepted JSON summary.

In [ ]:
stored_path = result.output_dir / 'topology_metrics_summary.json'
stored = json.loads(stored_path.read_text(encoding='utf-8'))
stored_rows = stored['metrics'] if isinstance(stored, dict) and 'metrics' in stored else stored
calculated = {row['metric_name']: row['value'] for row in result.calculation.summary_rows}
accepted = {row['metric_name']: row['value'] for row in stored_rows}
assert calculated == accepted
selected_names = ['node_count','physical_edge_count','connected_component_count','degree_mean','dead_end_count','degree_3_count','degree_4_count','cycle_rank','bridge_count','articulation_point_count','node_connectivity','edge_connectivity','average_shortest_path_hops','diameter_hops','radius_hops','average_shortest_path_length_local']
for name in selected_names:
    print(f'{name}: {calculated[name]}')
print('ENV38_TOPOLOGY_STORED_CROSSCHECK: PASS')

## 14. Existing figures
Topology detail is communicated through the accepted tabular artifacts; integrated notebook 08 displays the canonical-network and degree-structure figures without inventing a new visualization pipeline.

## 15. Artifact inventory and provenance
The run directory contains summary JSON/CSV, node and edge tables, the OSMnx cross-check, methodology, and a manifest bound to canonical identity and software provenance.

In [ ]:
required = ['topology_metrics_summary.json','topology_metrics_summary.csv','node_topology_metrics.csv','edge_topology_metrics.csv','osmnx_crosscheck.json','topology_methodology.json','topology_analysis_manifest.json']
assert all((result.output_dir / name).is_file() for name in required)
print('ENV38_TOPOLOGY_ARTIFACTS: ' + ', '.join(required))

## 16. Limitations and reproducibility summary
This is a descriptive, offline analysis of one accepted local-coordinate network. It performs no download, geocoding, projection, nearest-node search, routing service call, or causal comparison. The notebook uses production calculations with `publish=False`, so accepted files remain unchanged.

In [ ]:
print(f'ACCEPTED_ENV38_RUN: {result.selection.run_id}')
print(f'TOPOLOGY_METRICS: {len(result.calculation.summary_rows)}')
print('NOTEBOOK_06_EXECUTION: PASS')